[!["Open In Colab"](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ContextLab/llm-course/blob/main/slides/week9/moe_efficiency_demo.ipynb)

# Mixture of experts and efficiency — hands-on exploration

**PSYC 51.17: Models of language and communication**  
**Week 9**

---

## Learning objectives

By the end of this session, you will:
1. Implement a Mixture of Experts (MoE) layer from scratch in PyTorch
2. Visualize and diagnose the "routing collapse" problem in sparse models
3. Apply INT8 quantization to a neural network and measure memory savings
4. Compare the performance of dense vs. sparse models under a fixed compute budget

## Setup

In [ ]:
# Install required packages (for Colab)
!pip install -q torch matplotlib numpy

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import time
import warnings
warnings.filterwarnings('ignore')

print("\u2713 All imports successful!")

## Part 1: Building a Mixture of Experts layer

A **Mixture of Experts (MoE)** layer replaces a single large feed-forward network with multiple smaller "experts." A **router** decides which experts should process each token. This allows us to scale the model's capacity (total parameters) without increasing the compute cost per token (active parameters).

In [ ]:
class MoELayer(nn.Module):
    def __init__(self, d_model, num_experts=8, k=2):
        super().__init__()
        self.num_experts = num_experts
        self.k = k
        self.gate = nn.Linear(d_model, num_experts)
        
        # Each expert is a small MLP
        self.experts = nn.ModuleList([
            nn.Sequential(
                nn.Linear(d_model, 4 * d_model),
                nn.GELU(),
                nn.Linear(4 * d_model, d_model)
            ) for _ in range(num_experts)
        ])

    def forward(self, x):
        # x shape: (batch_size, seq_len, d_model)
        batch_size, seq_len, d_model = x.shape
        x_flat = x.view(-1, d_model)  # (total_tokens, d_model)
        
        # Step 1: Compute routing scores
        router_logits = self.gate(x_flat)
        router_probs = F.softmax(router_logits, dim=-1)
        
        # Step 2: Select top-k experts
        top_k_probs, top_k_idx = torch.topk(router_probs, self.k, dim=-1)
        
        # Step 3: Normalize selected weights
        top_k_probs = top_k_probs / top_k_probs.sum(dim=-1, keepdim=True)
        
        # Step 4: Route tokens to experts and combine outputs
        output = torch.zeros_like(x_flat)
        
        for i in range(self.k):
            # For each of the k slots, find which tokens go to which expert
            expert_indices = top_k_idx[:, i]
            probs = top_k_probs[:, i]
            
            for eid in range(self.num_experts):
                mask = (expert_indices == eid)
                if mask.any():
                    # Process tokens assigned to this expert
                    expert_output = self.experts[eid](x_flat[mask])
                    output[mask] += probs[mask].unsqueeze(-1) * expert_output
                    
        return output.view(batch_size, seq_len, d_model), top_k_idx

# Initialize MoE layer
d_model = 128
moe = MoELayer(d_model=d_model, num_experts=8, k=2)

# Create random input (batch=1, seq_len=10)
x = torch.randn(1, 10, d_model)
output, selected_experts = moe(x)

print(f"Input shape: {x.shape}")
print(f"Output shape: {output.shape}")
print("\nSelected experts for each token (top-2):")
for i, experts in enumerate(selected_experts):
    print(f"  Token {i}: Expert {experts[0].item()} and Expert {experts[1].item()}")

In [ ]:
# Visualize routing distribution
seq_len = 20
x_viz = torch.randn(1, seq_len, d_model)
_, top_k_idx = moe(x_viz)

# Create a heatmap of expert selection
routing_map = np.zeros((seq_len, 8))
for i in range(seq_len):
    for eid in top_k_idx[i]:
        routing_map[i, eid.item()] = 1

plt.figure(figsize=(10, 6))
plt.imshow(routing_map, cmap="Greens", aspect="auto")
plt.title("Expert Routing Heatmap (Tokens \u00d7 Experts)", fontsize=14)
plt.xlabel("Expert ID", fontsize=12)
plt.ylabel("Token Index", fontsize=12)
plt.xticks(range(8))
plt.yticks(range(seq_len))
plt.colorbar(label="Selected (Top-2)")
plt.tight_layout()
plt.show()

In [ ]:
# Compare compute: Dense FFN vs MoE
def count_parameters(model):
    return sum(p.numel() for p in model.parameters())

dense_ffn = nn.Sequential(
    nn.Linear(d_model, 4 * d_model),
    nn.GELU(),
    nn.Linear(4 * d_model, d_model)
)

print(f"Dense FFN Total Parameters:  {count_parameters(dense_ffn):,}")
print(f"MoE Layer Total Parameters:  {count_parameters(moe):,}")
print(f"MoE Active Parameters/Token: {count_parameters(moe.experts[0]) * 2 + count_parameters(moe.gate):,}")

# FLOPs estimation (simplified: weights * 2 for multiply-add)
dense_flops = count_parameters(dense_ffn) * 2
moe_active_flops = (count_parameters(moe.experts[0]) * 2 + count_parameters(moe.gate)) * 2

print(f"\nDense FLOPs per token:       {dense_flops:,}")
print(f"MoE Active FLOPs per token:  {moe_active_flops:,}")

### 💡 Discussion

- Why do we normalize the `top_k_probs`? What would happen if we didn't?
- In our implementation, we use a loop over experts. Why is this inefficient for hardware like GPUs, and how might production libraries (like Megatron-LM) handle this?
- If we have 64 experts but only use top-2, what is the ratio of total parameters to active parameters?
- How does the router "know" which expert is good at which task?

## Part 2: The load balancing problem

A major challenge in MoE training is **routing collapse**: the router learns to send almost all tokens to just one or two experts. These experts get all the gradient updates and become even better, while the other experts stay "dead." This wastes the model's capacity.

In [ ]:
# Simulate routing collapse
num_tokens = 1000
x_large = torch.randn(num_tokens, 1, d_model)
_, top_k_idx = moe(x_large)

expert_counts = torch.zeros(8)
for idx in top_k_idx.view(-1):
    expert_counts[idx] += 1

plt.figure(figsize=(10, 5))
plt.bar(range(8), expert_counts.numpy(), color="#267aba")
plt.title("Expert Selection Frequency (Untrained Router)", fontsize=14)
plt.xlabel("Expert ID", fontsize=12)
plt.ylabel("Number of times selected", fontsize=12)
plt.xticks(range(8))
plt.grid(axis='y', alpha=0.3)
plt.show()

In [ ]:
def compute_aux_loss(router_probs, num_experts):
    """
    Simplified auxiliary load-balancing loss.
    Encourages uniform distribution of tokens across experts.
    """
    # router_probs: (total_tokens, num_experts)
    # f: fraction of tokens routed to each expert
    # P: average probability assigned to each expert
    
    f = router_probs.mean(dim=0)
    P = router_probs.mean(dim=0)
    
    # Loss is proportional to the dot product of f and P
    # Minimized when both are uniform (1/N)
    aux_loss = num_experts * torch.sum(f * P)
    return aux_loss

# Example of training the router to balance
optimizer = torch.optim.Adam(moe.gate.parameters(), lr=0.01)

print("Training router with auxiliary loss...")
for i in range(100):
    optimizer.zero_grad()
    x_batch = torch.randn(128, d_model)
    logits = moe.gate(x_batch)
    probs = F.softmax(logits, dim=-1)
    
    loss = compute_aux_loss(probs, 8)
    loss.backward()
    optimizer.step()
    
    if (i+1) % 20 == 0:
        print(f"  Step {i+1}: Aux Loss = {loss.item():.4f}")

# Check balance after training
_, top_k_idx_balanced = moe(x_large)
balanced_counts = torch.zeros(8)
for idx in top_k_idx_balanced.view(-1):
    balanced_counts[idx] += 1

plt.figure(figsize=(10, 5))
plt.bar(range(8), balanced_counts.numpy(), color="#00693e")
plt.title("Expert Selection Frequency (After Load Balancing)", fontsize=14)
plt.xlabel("Expert ID", fontsize=12)
plt.ylabel("Number of times selected", fontsize=12)
plt.xticks(range(8))
plt.grid(axis='y', alpha=0.3)
plt.show()

### 💡 Discussion

- Why is a perfectly balanced distribution desirable? Is there any case where we might *want* some experts to be used more than others?
- What happens if the auxiliary loss coefficient is too high? Could it hurt the model's performance?
- How does "expert capacity" (limiting the number of tokens per expert) differ from an auxiliary loss?
- If an expert is "dead," how can we bring it back to life during training?

## Part 3: Quantization in practice

**Quantization** reduces the precision of model weights (e.g., from 32-bit floats to 8-bit integers). This slashes memory usage and can speed up inference on hardware that supports integer arithmetic.

In [ ]:
# Create a simple model
model = nn.Sequential(
    nn.Linear(10, 100),
    nn.ReLU(),
    nn.Linear(100, 2)
)

def quantize_to_int8(tensor):
    """Naive symmetric quantization to INT8."""
    # 1. Find the scale factor
    max_val = tensor.abs().max().item()
    scale = 127 / max_val
    
    # 2. Scale, round, and clip
    quantized = torch.round(tensor * scale).clamp(-128, 127).to(torch.int8)
    
    # 3. Store dequantization scale
    return quantized, scale

def dequantize(quantized, scale):
    return quantized.to(torch.float32) / scale

# Quantize the first layer's weights
original_weights = model[0].weight.data.clone()
quantized_weights, scale = quantize_to_int8(original_weights)
recovered_weights = dequantize(quantized_weights, scale)

print(f"Original weight (first 5):  {original_weights[0, :5]}")
print(f"Quantized weight (first 5): {quantized_weights[0, :5]}")
print(f"Recovered weight (first 5): {recovered_weights[0, :5]}")

# Memory savings
original_size = original_weights.nelement() * 4  # 4 bytes for FP32
quantized_size = quantized_weights.nelement() * 1  # 1 byte for INT8
print(f"\nOriginal memory:  {original_size} bytes")
print(f"Quantized memory: {quantized_size} bytes")
print(f"Savings:          {original_size / quantized_size:.1f}x")

In [ ]:
# Visualize weight distributions
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.hist(original_weights.view(-1).numpy(), bins=50, color="#267aba", alpha=0.7)
plt.title("Original Weights (FP32)", fontsize=13)
plt.xlabel("Value")
plt.ylabel("Frequency")

plt.subplot(1, 2, 2)
plt.hist(quantized_weights.view(-1).numpy(), bins=50, color="#9d162e", alpha=0.7)
plt.title("Quantized Weights (INT8)", fontsize=13)
plt.xlabel("Integer Value (-128 to 127)")
plt.ylabel("Frequency")

plt.tight_layout()
plt.show()

In [ ]:
# Compare predictions
x_test = torch.randn(1, 10)
with torch.no_grad():
    # Original prediction
    original_out = model(x_test)
    
    # Prediction with quantized weights in first layer
    model[0].weight.data = recovered_weights
    quantized_out = model(x_test)

print(f"Original output:  {original_out}")
print(f"Quantized output: {quantized_out}")
print(f"Difference:       {torch.norm(original_out - quantized_out).item():.6f}")

### 💡 Discussion

- Why did the output change slightly? What is "quantization error"?
- We used "symmetric" quantization (centered at zero). What is "asymmetric" quantization, and when might it be better?
- Why is INT4 quantization so popular for LLMs today? What is the tradeoff between bits and accuracy?
- Can we train a model *while* it is quantized? (Look up Quantization-Aware Training or QAT).

## Part 4: Comparing dense vs sparse

Does MoE actually help? Let's compare a dense model and an MoE model with the **same compute budget** (active parameters) on a simple task: predicting the next character in a short text.

In [ ]:
# Simple text data
text = "mixture of experts is an efficient way to scale neural networks. " * 50
chars = sorted(list(set(text)))
char_to_idx = {ch: i for i, ch in enumerate(chars)}
idx_to_char = {i: ch for i, ch in enumerate(chars)}
data = [char_to_idx[ch] for ch in text]

def get_batch(batch_size=32, seq_len=16):
    ix = torch.randint(len(data) - seq_len, (batch_size,))
    x = torch.stack([torch.tensor(data[i:i+seq_len]) for i in ix])
    y = torch.stack([torch.tensor(data[i+1:i+seq_len+1]) for i in ix])
    return x, y

class SimpleModel(nn.Module):
    def __init__(self, vocab_size, d_model, use_moe=False):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        if use_moe:
            # MoE with 4 experts, top-1 routing
            # Each expert is same size as the dense FFN
            self.layer = MoELayer(d_model, num_experts=4, k=1)
        else:
            # Dense FFN
            self.layer = nn.Sequential(
                nn.Linear(d_model, 4 * d_model),
                nn.GELU(),
                nn.Linear(4 * d_model, d_model)
            )
        self.head = nn.Linear(d_model, vocab_size)
        self.use_moe = use_moe

    def forward(self, x):
        x = self.embed(x)
        if self.use_moe:
            x, _ = self.layer(x)
        else:
            x = self.layer(x)
        return self.head(x)

vocab_size = len(chars)
d_model = 64

dense_model = SimpleModel(vocab_size, d_model, use_moe=False)
moe_model = SimpleModel(vocab_size, d_model, use_moe=True)

print(f"Dense Model Total Params: {count_parameters(dense_model):,}")
print(f"MoE Model Total Params:   {count_parameters(moe_model):,}")
print(f"Both use ~{count_parameters(dense_model):,} active params per token.")

In [ ]:
def train(model, steps=200):
    optimizer = torch.optim.Adam(model.parameters(), lr=0.005)
    losses = []
    for i in range(steps):
        x, y = get_batch()
        logits = model(x)
        loss = F.cross_entropy(logits.view(-1, vocab_size), y.view(-1))
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
    return losses

print("Training Dense Model...")
dense_losses = train(dense_model)

print("Training MoE Model...")
moe_losses = train(moe_model)

plt.figure(figsize=(10, 6))
plt.plot(dense_losses, label="Dense (Baseline)", color="#267aba", alpha=0.8)
plt.plot(moe_losses, label="MoE (Sparse)", color="#00693e", linewidth=2)
plt.title("Training Loss: Dense vs. MoE (Matched Compute)", fontsize=14)
plt.xlabel("Step", fontsize=12)
plt.ylabel("Cross Entropy Loss", fontsize=12)
plt.legend()
plt.grid(alpha=0.3)
plt.show()

### 💡 Discussion

- Why does the MoE model reach a lower loss even though it uses the same amount of compute per token?
- If we kept increasing the number of experts, would the loss keep going down? What is the limit?
- In this toy example, the MoE model has more total parameters. In a real-world scenario, where does this extra memory cost become a problem?
- How does this relate to the "Scaling Laws" we discussed in Lecture 16?

## Summary

| Concept | What we did | Key insight |
|---------|-------------|-------------|
| **MoE Layer** | Built a router + experts | Sparse activation scales capacity without scaling compute |
| **Load Balancing** | Visualized routing collapse | Routers need help (aux loss) to use all experts effectively |
| **Quantization** | Converted FP32 to INT8 | Lower precision saves 4x memory with minimal error |
| **Dense vs Sparse** | Compared loss curves | MoE outperforms dense models at the same compute budget |

## Further exploration

1. **Expert Specialization**: Modify the training task to include two different languages or tasks (e.g., math vs. text). Can you visualize the router learning to send math tokens to one expert and text tokens to another?
2. **Quantization-Aware Training**: Try adding a small amount of noise to the weights during training to simulate quantization. Does this make the model more robust to INT8 conversion?
3. **Mamba vs. Attention**: Research the Mamba architecture. How does its linear scaling compare to the quadratic scaling of the attention mechanism we used here?
4. **Distillation**: Take your trained MoE model and try to "distill" its knowledge into a smaller dense model. How much of the performance can you keep?